## Our Framework Evaluation with Wisig Dataset

Dataset: [link](https://cores.ee.ucla.edu/downloads/datasets/wisig/)

In [26]:
%reload_ext autoreload
%autoreload 2

import os
import numpy as np
import matlab
import cfo_utils
import utils
from sklearn.utils import shuffle
from scipy import sparse
import matplotlib.pyplot as plt
from dataset_api import DatasetAPI
from evaluation_api import EvaluationAPI
from fingerprinting_api import FingerprintingAPI
from dataset_preparation import ChannelIndSpectrogram
import random
import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras import regularizers
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import *
import tensorflow.keras.backend as K


random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)
os.environ['TF_DETERMINISTIC_OPS'] = '1'

# DATASET_NAME =           DatasetAPI.DATASET_WISIG_OLD
DATASET_NAME =           DatasetAPI.DATASET_WISIG_NEW
RX_NODES =               [DatasetAPI.RX_1]
ROOT_DIRECTORY =         '/home/smazokha2016/Desktop'
MATLAB_SRC_DIRECTORY =   '/home/smazokha2016/Desktop/mobintel-rffi/fingerprinting'
MATLAB_SESSION_ID =      'fp_workflow'
AUG_ON =                 False

DATA_CONFIG = {
    'dataset_name':      DATASET_NAME,
    'frame_count_train': 1000, # 200 for v2, 500 for v4, 500 for wisig
    'frame_count_epoch': 100,
    'samples_count':     400
}

AUG_CONFIG = {
    'multiplier':        5,
    't_rms_bounds':      matlab.double([1, 2]),
    'd_f_bounds':        matlab.double([0, 1]),
    'k_factor_bounds':   matlab.double([0, 1]),
    'awgn':              matlab.double([0, 1]),
}

In [27]:
# 1. Let's create the classifier model
def create_net(n_tx, show_summary=False):
    inputs = Input(shape=(256,2))
    x = Reshape((256,2,1))(inputs)
    x = Conv2D(8,(3,2),activation='relu',padding = 'same')(x)
    x = MaxPool2D((2,1))(x)
    x = Conv2D(16,(3,2),activation='relu',padding = 'same')(x)
    x = MaxPool2D((2,1))(x)
    x = Conv2D(16,(3,2),activation='relu',padding = 'same')(x)
    x = MaxPool2D((2,2))(x)
    x = Conv2D(32,(3,1),activation='relu',padding = 'same')(x)
    x = MaxPool2D((2,1))(x)
    x = Conv2D(16,(3,1),activation='relu',padding = 'same')(x)
    #x = resnet(x,64,(3,2),'6')
    #x = MaxPool2D((2,2))(x)
    x = Flatten()(x)

    x = Dense(100, activation='relu', kernel_regularizer = keras.regularizers.l2(0.0001))(x)
    # x = Dropout(0.3)(x)
    x = Dense(80, activation='relu',kernel_regularizer = keras.regularizers.l2(0.0001))(x)
    x = Dropout(0.5)(x)
    x = Dense(n_tx, activation='softmax',kernel_regularizer = keras.regularizers.l2(0.0001))(x)
    ops = x

    classifier = Model(inputs,ops)
    classifier.compile(loss='categorical_crossentropy',metrics=['categorical_accuracy'],optimizer=keras.optimizers.Adam(0.0005))

    if show_summary: print(classifier.summary())
    
    return classifier

In [71]:
def convert_data_complex_to_real(data):
    data_real = np.zeros((data.shape[0], data.shape[1], 2))
    data_real[:, :, 0] = np.real(data)
    data_real[:, :, 1] = np.imag(data)
    return data_real

def load_wisig_classification_data(dataset_api, first_day, ndays, nsamples, augment, augment_multiplier, devices_disjoint, equalized):
    dataset_paths, _, _, node_ids_epoch, _ = dataset_api.load_dataset_info(dataset_name=DatasetAPI.DATASET_WISIG_NEW, rx_name='node1-1', allowed_epochs=None, wisig_equalized=equalized, wisig_disjoint=devices_disjoint)

    data_train = np.zeros((0, nsamples), dtype=complex)
    data_validate = np.zeros((0, nsamples), dtype=complex)
    data_test = np.zeros((0, nsamples), dtype=complex)
    labels_train = np.zeros((0, 1))
    labels_validate = np.zeros((0, 1))
    labels_test = np.zeros((0, 1))

    for day in range(first_day, first_day + ndays):
        # Retrieve data and labels for a given day
        data_day, label_day, _ = dataset_api.load_raw_dataset(dataset_paths[day], shuffle=False)
        data_day = data_day[:, 0:nsamples]
        print(f'Data raw: {data_day.shape}')

        # Filter and keep only the device IDs and frames that we decided to use for training
        data_day_train, label_day_train, _ = dataset_api.filter_dataset(data_day, label_day, None, node_ids_epoch, np.arange(0, 400))
        data_day_validate, label_day_validate, _ = dataset_api.filter_dataset(data_day, label_day, None, node_ids_epoch, np.arange(400, 450))
        data_day_test, label_day_test, _ = dataset_api.filter_dataset(data_day, label_day, None, node_ids_epoch, np.arange(450, 500))

        if augment:
            data_day_train, label_day_train, _ = dataset_api.augment_dataset(data_day_train, label_day_train, None, augment_cfo=False, multiplier=augment_multiplier)

        # Add day's data to the unified arrays
        data_train = np.concatenate((data_train, data_day_train), axis=0)
        data_validate = np.concatenate((data_validate, data_day_validate), axis=0)
        data_test = np.concatenate((data_test, data_day_test), axis=0)
        labels_train = np.concatenate((labels_train, label_day_train), axis=0)
        labels_validate = np.concatenate((labels_validate, label_day_validate), axis=0)
        labels_test = np.concatenate((labels_test, label_day_test), axis=0)

    data_train, labels_train = shuffle(data_train, labels_train, random_state = 42)

    # Convert labels to integers
    labels_train_categorical, labels_train_weights = sparse_to_categorical(labels_train.astype(int))
    labels_validate_categorical, _ = sparse_to_categorical(labels_validate.astype(int))
    labels_test_categorical, _ = sparse_to_categorical(labels_test.astype(int))

    # Convert data from complex to real and complex subarrays
    data_train = convert_data_complex_to_real(data_train)
    data_validate = convert_data_complex_to_real(data_validate)
    data_test = convert_data_complex_to_real(data_test)

    print(f'Final data: {data_train.shape}, {data_validate.shape}, {data_test.shape}')

    return data_train, labels_train_categorical, data_validate, labels_validate_categorical, data_test, labels_test_categorical, labels_train_weights

def convert_weights_to_dict(weights):
    return {i: weight for i, weight in enumerate(weights)}

def sparse_to_categorical(labels):
    # Convert vector of sparse labels to a one-hot encoded 2D array
    labels = np.asarray(labels)
    unique_labels = np.unique(labels)
    label_to_idx = {int(label): idx for idx, label in enumerate(unique_labels)}
    labels_int = labels.astype(int)
    labels_categorical = np.zeros((len(labels), len(unique_labels)), dtype=int)
    labels_categorical[np.arange(len(labels)), [label_to_idx[int(label)] for label in labels_int]] = 1

    # Calculate weights for each of the categories
    category_sum = np.sum(labels_categorical, axis=0)
    labels_weights = (np.max(category_sum, axis=0)/category_sum).tolist()
    return labels_categorical, labels_weights

def train_classifier(sig_train, txid_train, sig_valid, txid_valid, cls_weights, n_tx, n_epochs):
    classifier = create_net(n_tx, show_summary=False)
    filepath = 't_weights_1'
    c = [
        keras.callbacks.ModelCheckpoint(filepath, monitor='val_loss', verbose=1, save_best_only=True),
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)]
    
    class_weight_dict = convert_weights_to_dict(cls_weights)

    print(sig_train.shape)
    print(txid_train.shape)

    history = classifier.fit(sig_train, txid_train, class_weight=class_weight_dict, validation_data=(sig_valid, txid_valid), callbacks=c, epochs=n_epochs)

    return classifier

def evaluate_model(dataset_api, ndays, augment, augment_multiplier, devices_disjoint, equalized):
    nsamples = 256
    n_epochs = 100

    # Retrieve combined training and validation data from ndays
    data_train, labels_train, data_valid, labels_valid, _, _, train_weights = load_wisig_classification_data(dataset_api, 
                                                                                                             first_day=0, ndays=ndays, nsamples=nsamples, 
                                                                                                             augment=augment, augment_multiplier=augment_multiplier, 
                                                                                                             devices_disjoint=devices_disjoint, equalized=equalized)

    # Retrieve testing data from day 1
    data_test_sameday, labels_test_sameday, _, _, _, _, _ = load_wisig_classification_data(dataset_api, 
                                                                                        first_day=0, ndays=1, nsamples=nsamples, 
                                                                                        augment=augment, augment_multiplier=augment_multiplier, 
                                                                                        devices_disjoint=devices_disjoint, equalized=equalized)

    # Retrieve testing data from day 4
    data_test_diffday, labels_test_diffday, _, _, _, _, _ = load_wisig_classification_data(dataset_api, 
                                                                                        first_day=3, ndays=1, nsamples=nsamples, 
                                                                                        augment=augment, augment_multiplier=augment_multiplier, 
                                                                                        devices_disjoint=devices_disjoint, equalized=equalized)



    # Train the classifier model
    classifier = train_classifier(data_train, labels_train, data_valid, labels_valid, train_weights, len(train_weights), n_epochs)

    # Evaluate the classifier model
    results_sameday = classifier.evaluate(data_test_sameday, labels_test_sameday, verbose=0)[1]
    results_diffday = classifier.evaluate(data_test_diffday, labels_test_diffday, verbose=0)[1]

    print('Results for same day: {}'.format(results_sameday))
    print('Results for diff day: {}'.format(results_diffday))

    return results_sameday, results_diffday

dataset_api = DatasetAPI(root_dir=ROOT_DIRECTORY, matlab_src_dir=MATLAB_SRC_DIRECTORY, matlab_session_id=MATLAB_SESSION_ID, aug_on=AUG_ON)

# Dimensions: ([sameday | diffday], [1 | 1+2 | 1+2+3], [raw | equalized])
wisig_classifier_accuracies = np.zeros((2, 3, 2))

# for i, equalized in enumerate([False, True]):
for i, equalized in enumerate([True]):
    for ndays in range(2, 4):
        print(f'Evaluating WiSig classifier: ndays={ndays}, equalized={equalized}')
        accuracy_sameday, accuracy_diffday = evaluate_model(dataset_api, ndays=ndays, augment=True, augment_multiplier=2, devices_disjoint=False, equalized=equalized)
        wisig_classifier_accuracies[:, ndays-1, i] = [accuracy_sameday, accuracy_diffday]

Evaluating WiSig classifier: ndays=2, equalized=True
Data raw: (22000, 256)
Augmenting dataset: x2, randomized CFO adding disabled.
Data raw: (21000, 256)
Augmenting dataset: x2, randomized CFO adding disabled.
Final data: (9600, 256, 2), (600, 256, 2), (600, 256, 2)


/tmp/ipykernel_2709724/2470811350.py:65: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  labels_categorical[np.arange(len(labels)), [label_to_idx[int(label)] for label in labels_int]] = 1


Data raw: (22000, 256)
Augmenting dataset: x2, randomized CFO adding disabled.
Final data: (4800, 256, 2), (300, 256, 2), (300, 256, 2)
Data raw: (20500, 256)
Augmenting dataset: x2, randomized CFO adding disabled.
Final data: (4800, 256, 2), (300, 256, 2), (300, 256, 2)
(9600, 256, 2)
(9600, 6)
Epoch 1/100
299/300 [============================>.] - ETA: 0s - loss: 0.8675 - categorical_accuracy: 0.6438
Epoch 1: val_loss improved from inf to 0.15741, saving model to t_weights_1
INFO:tensorflow:Assets written to: t_weights_1/assets


INFO:tensorflow:Assets written to: t_weights_1/assets


300/300 [==============================] - 4s 9ms/step - loss: 0.8654 - categorical_accuracy: 0.6449 - val_loss: 0.1574 - val_categorical_accuracy: 0.9700
Epoch 2/100
296/300 [============================>.] - ETA: 0s - loss: 0.1556 - categorical_accuracy: 0.9537
Epoch 2: val_loss improved from 0.15741 to 0.04581, saving model to t_weights_1
INFO:tensorflow:Assets written to: t_weights_1/assets


...

450/450 [==============================] - 3s 7ms/step - loss: 0.0027 - categorical_accuracy: 0.9999 - val_loss: 0.0022 - val_categorical_accuracy: 1.0000
Epoch 77/100
444/450 [============================>.] - ETA: 0s - loss: 0.0027 - categorical_accuracy: 0.9998
Epoch 77: val_loss improved from 0.00216 to 0.00203, saving model to t_weights_1
INFO:tensorflow:Assets written to: t_weights_1/assets


INFO:tensorflow:Assets written to: t_weights_1/assets


450/450 [==============================] - 3s 7ms/step - loss: 0.0026 - categorical_accuracy: 0.9998 - val_loss: 0.0020 - val_categorical_accuracy: 1.0000
Epoch 78/100
448/450 [============================>.] - ETA: 0s - loss: 0.0026 - categorical_accuracy: 0.9997
Epoch 78: val_loss improved from 0.00203 to 0.00199, saving model to t_weights_1
INFO:tensorflow:Assets written to: t_weights_1/assets


INFO:tensorflow:Assets written to: t_weights_1/assets


450/450 [==============================] - 3s 7ms/step - loss: 0.0026 - categorical_accuracy: 0.9997 - val_loss: 0.0020 - val_categorical_accuracy: 1.0000
Epoch 79/100
446/450 [============================>.] - ETA: 0s - loss: 0.0042 - categorical_accuracy: 0.9994
Epoch 79: val_loss did not improve from 0.00199
450/450 [==============================] - 3s 6ms/step - loss: 0.0042 - categorical_accuracy: 0.9994 - val_loss: 0.0021 - val_categorical_accuracy: 1.0000
Epoch 80/100
442/450 [============================>.] - ETA: 0s - loss: 0.0040 - categorical_accuracy: 0.9996
Epoch 80: val_loss did not improve from 0.00199
450/450 [==============================] - 2s 5ms/step - loss: 0.0040 - categorical_accuracy: 0.9996 - val_loss: 0.0022 - val_categorical_accuracy: 1.0000
Epoch 81/100
441/450 [============================>.] - ETA: 0s - loss: 0.0024 - categorical_accuracy: 1.0000
Epoch 81: val_loss did not improve from 0.00199
450/450 [==============================] - 2s 5ms/step - loss

In [70]:
# Dimensions: ([sameday | diffday], [1 | 1+2 | 1+2+3], [raw | equalized])
wisig_classifier_accuracies

array([[[0., 0.],
        [0., 0.],
        [0., 0.]],

       [[0., 0.],
        [0., 0.],
        [0., 0.]]])